In [1]:
import pandas as pd
import pedpy
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import glob
from dataclasses import dataclass, field
from typing import List
import os
from scipy.interpolate import interp1d

In [7]:
"""Module containing functions to compute pair distribution function."""

import warnings
from typing import Tuple

import numpy as np
import numpy.typing as npt
import pandas
from scipy.spatial.distance import cdist

from pedpy.column_identifier import FRAME_COL, ID_COL, X_COL, Y_COL
from pedpy.data.trajectory_data import TrajectoryData


def compute_pair_distibution_function(
    *,
    traj_data: TrajectoryData,
    radius_bin_size: float,
    randomisation_stacking: int = 1,
) -> Tuple[npt.NDArray[np.float16], npt.NDArray[np.float16]]:
    """
    Computes the pair distribution function g(r) for a given set of trajectory data.

    This function calculates the spatial distribution of positions :math:`g(r)`
    For a variable :math:`r`, the pdf is given by the probability that two pedestrians are separated
    by :math:`r` normalized by the probability :math:`PNI(r)` that two non-interacting pedestrians
    are separated by :math:`r`, specifically

    .. math::
        g(r) = P(r)/PNI(r),

    Args:
    - traj_data: TrajectoryData, an object containing the trajectories.
    - radius_bin_size: float, the size of the bins for the radial distribution function in the same units as the positions.
    - randomisation_stacking: int, Number of time the dataset will be stacked before being randomly shuffled to exact distances of non-interacting pedestrians. Larger stacking number will lead to closer approximation of true pairwise distribution of non-interacting pedestrians but with also increase computation cost.


    Returns:
    - Tuple[np.ndarray, np.ndarray]: A tuple of two numpy arrays. The first array contains the bin edges (excluding the first bin edge),
      and the second array contains the values of the pair distribution function :math:`g(r)` for each bin.
    """
    df = traj_data.data

    # Create Dataframe with all pairwise distances
    dist_pd_array = calculate_data_frame_pair_dist(df)

    # Concatenate the working dataframe df  to match the number of randomization cycles
    concatenated_random_df = pandas.concat(
        [df] * randomisation_stacking, ignore_index=True
    )
    # Scramble time-information to mitigate finite-size effects and calculate pairwise distances of scrambled dataset
    concatenated_random_df.frame = concatenated_random_df.frame.sample(
        frac=1
    ).reset_index(drop=True)
    dist_pd_ni_array = calculate_data_frame_pair_dist(concatenated_random_df)

    ## Create the bin for data
    radius_bins = np.arange(0, dist_pd_array.max(), radius_bin_size)

    # Calculate pair distibution: g(r)
    ## Actual distribution
    pd_bins = pandas.cut(dist_pd_array, radius_bins)
    pd_bins_normalised = (pd_bins.value_counts().sort_index().to_numpy()) / len(
        dist_pd_array
    )  # Normalising by the number of pairwise distances in the dataframe
    ## Scrambled distribution
    pd_ni_bins = pandas.cut(dist_pd_ni_array, radius_bins)
    pd_ni_bins_normalised = (
        pd_ni_bins.value_counts().sort_index().to_numpy()
    ) / len(
        dist_pd_ni_array
    )  # Normalising by the number of pairwise distances in the dataframe

    # Suppress warnings
    warnings.filterwarnings("ignore")

    try:
        with np.errstate(divide="raise"):
            pair_distribution = pd_bins_normalised / pd_ni_bins_normalised
        warnings.filterwarnings("default")  # reset wrnings values

    except FloatingPointError:
        warnings.filterwarnings("default")  # reset wrnings values
        pair_distribution = pd_bins_normalised / pd_ni_bins_normalised
        warning_message = "Random probability distribution contains null values, try using larger dx or more randomization cycles."
        warnings.warn(warning_message)

    return radius_bins[1:], pair_distribution


def calculate_data_frame_pair_dist_diff_gender(
    df: pandas.DataFrame,
    male_identifier: int = 2,
    female_identifier: int = 1,
) -> npt.NDArray[np.float16]:
    distances_list = []
    GENDER_COL = 'gender'
    for _, frame_df in df.groupby(FRAME_COL):
        male_df = frame_df[frame_df[GENDER_COL] == male_identifier]
        female_df = frame_df[frame_df[GENDER_COL] == female_identifier]

        if not male_df.empty and not female_df.empty:
            male_coordinates = male_df[[X_COL, Y_COL]].values
            female_coordinates = female_df[[X_COL, Y_COL]].values

            # Calculate pairwise distances from each male to each female in the current frame
            frame_distances = cdist(male_coordinates, female_coordinates, metric="euclidean")

            # Since we're only interested in male to female distances, no need to extract upper triangle
            distances_list.extend(frame_distances.flatten())

    return np.array(distances_list, dtype=np.float16)


def calculate_data_frame_pair_dist(
    df: pandas.DataFrame,
) -> npt.NDArray[np.float16]:
    distances_list = []

    for _, frame_df in df.groupby(FRAME_COL):
        unique_ids = frame_df[ID_COL].unique()
        N_ids = len(unique_ids)
        if N_ids > 1:
            x_values = frame_df[X_COL].values
            y_values = frame_df[Y_COL].values
            coordinates = np.stack((x_values, y_values), axis=-1)
            # Calculate pairwise distances for the current frame using cdist
            frame_distances = cdist(
                coordinates, coordinates, metric="euclidean"
            )

            # Extract the upper triangle without the diagonal
            distances_upper_triangle = frame_distances[
                np.triu_indices_from(frame_distances, k=1)
            ]

            distances_list.extend(distances_upper_triangle)

    return np.array(distances_list)


In [2]:
def extract_category_from_filename(filename: str) -> str:
    """
    Extracts the category part from a given filename, taking into account
    special cases like 'mix_sorted' and 'mix_random'.
    
    The category is extracted based on its position in the filename, which is
    assumed to follow the pattern: '../data/country/category_restOfTheFilename.csv'.
    
    Parameters:
    - filename (str): The full path of the file.
    
    Returns:
    - str: The extracted category.
    """
    # Extract the basename of the file (e.g., 'female_41_21.csv')
    basename = os.path.basename(filename)
    
    # Split by underscores and reconstruct the category part
    parts = basename.split('_')
    if parts[0].startswith("mix"):
        # Handles 'mix_sorted' and 'mix_random'
        category = "_".join(parts[:2])
    else:
        # Handles 'female', 'male', etc.
        category = parts[0]
    
    return category


In [3]:
@dataclass
class CategoryInfo:
    country: str
    category: str
    max_number: int
    max_number_filename: str
    min_number: int
    min_number_filename: str


def get_category_info(
    country: str, category: str, files: dict
) -> CategoryInfo:
    max_number = None
    min_number = None
    max_number_filename = None
    min_number_filename = None

    for filename in files[country]:
        gender = extract_category_from_filename(filename)
        if category == gender:
            current_number = int(filename.split("_")[-2])
            if max_number is None or current_number > max_number:
                max_number = current_number
                max_number_filename = filename

            if min_number is None or current_number < min_number:
                min_number = current_number
                min_number_filename = filename

    return CategoryInfo(
        country=country,
        category=category,
        max_number=max_number,
        max_number_filename=max_number_filename,
        min_number=min_number,
        min_number_filename=min_number_filename,
    )

In [4]:
# load_file and constants
categories = ["female", "male", "mix_sorted", "mix_random"]
colors = {
    "aus": "blue",
    "ger": "black",
    "jap": "orange",
    "chn": "red",
    "pal": "green",
}
max_radius = 4
max_g = 4

x_from = -1.4
x_to = 1.4
y_from = -2
y_to = -1
rename_mapping = {
    "ID": "id",
    "t(s)": "time",
    "x(m)": "x",
    "y(m)": "y",
}
column_types = {
    "id": int,
    "gender": int,
    "time": float,
    "x": float,
    "y": float,
}
countries = [
    "aus",
    "ger",
    "jap",
    "chn",
    "pal",
]
files = {}
for country in countries:
    files[country] = glob.glob(f"../data/{country}/*.csv")


def load_file(filename: str, filter: bool = False) -> pedpy.TrajectoryData:
    def calculate_fps(data: pd.DataFrame) -> int:
        """Calculate fps based on the mean difference of the 'time' column."""
        mean_diff = data.groupby("id")["time"].diff().dropna().mean()
        return int(round(1 / mean_diff))

    def set_column_types(data: pd.DataFrame, col_types) -> pd.DataFrame:
        """Set the types of the dataframe columns based on the given column types."""
        # Ensure columns are in data before type casting
        valid_types = {
            col: dtype for col, dtype in col_types.items() if col in data.columns
        }
        return data.astype(valid_types)

    data = pd.read_csv(filename)
    data.rename(columns=rename_mapping, inplace=True)
    data = set_column_types(data, column_types)
    fps = calculate_fps(data)
    if filter:
        data = data[
            (data["x"] > x_from)
            & (data["x"] < x_to)
            & (data["y"] > y_from)
            & (data["y"] < y_to)
        ].dropna()
    return pedpy.TrajectoryData(data=data, frame_rate=fps)

# Plot all files per country

In [5]:
#import other_characterisation_methods as pp
import other_characterisation_methods as pp
def plot_all_files_per_country(countries):
    for country in countries:
        print(country)
        for category in categories[:1]:
            print(category)
            fig, ax = plt.subplots(figsize=(5, 5))
            has_files = False  # Track if there are files for the current category
            all_distributions = []
            original_radius_bins_list = []
            for filename in files[country]:
                current_number = int(filename.split("_")[-2])
                if current_number < 15:
                    #print(f"skip {filename}")
                    continue
                gender = extract_category_from_filename(filename)
                if category == gender:
                    has_files = True  # Found at least one file for this category
                    #print(f"Processing {filename}...")
                    # if country != "pal": filter = True
                    # else: filter = False
                    filter= False
                    traj = load_file(filename, filter=filter)
                    
                    radius_bins, pair_distribution = (
                        pp.compute_pair_distibution_function(
                            traj_data=traj, radius_bin_size=0.1
                        )
                    )
                    all_distributions.append(pair_distribution)
                    original_radius_bins_list.append(radius_bins)
                    ax.plot(
                        radius_bins,
                        pair_distribution,
                        linewidth=0.2,
                        color="gray",
                        alpha=0.9,
                        label=filename,
                    )

            if has_files:                
                common_radius_bins = np.linspace(0, max_radius, 100)
                interpolated_distributions = []  # Store interpolated distributions here
                for dist, original_bins in zip(all_distributions, original_radius_bins_list):
                    # Create an interpolation function for the current distribution
                    interp_func = interp1d(original_bins, dist, bounds_error=False, fill_value=0)                
                    # Interpolate to the common radius bins
                    interpolated_dist = interp_func(common_radius_bins)                
                    # Store the interpolated distribution
                    interpolated_distributions.append(interpolated_dist)

                # Now, interpolated_distributions should contain arrays all of the same length
                distributions_array = np.array(interpolated_distributions)
                # Calculate min and max across these distributions for plotting
                distribution_min = np.min(distributions_array, axis=0)
                distribution_max = np.max(distributions_array, axis=0)
                ax.fill_between(common_radius_bins, distribution_min, distribution_max, color="gray", alpha=0.2, label="Range of Distributions")
                ax.set_title(
                    f"{category.capitalize()} Pair Distribution in {country}",
                    fontsize=14,
                )
                ax.set_xlabel(r"$r\; /\; m$", fontsize=14)
                ax.set_ylabel(r"$g(r)$", fontsize=14)
                ax.tick_params(axis="both", which="major", labelsize=12)
                # ax.grid(True, which='both', linestyle='--', alpha=0.3)
                ax.set_xlim([-0.1, max_radius])
                ax.set_ylim([-0.1, max_g])
                # ax.legend(title='Data Set', fontsize=10)

                fignames = [f"country/png/pair_distribution_{country}_{category}.png",
                            f"country/pdf/pair_distribution_{country}_{category}.pdf"]
                for figname in fignames:
                    plt.savefig(figname)
                plt.close(fig)
            else:
                plt.close(fig)  # Close the figure as it's unused
                print(f"No files found for {country} in the {category} category.")

    return traj.data
df = plot_all_files_per_country(countries=countries[1:2])

ger
female


## Plot per category at specific densities

In [6]:
def plot_category_info_min_max(countries, categories, files, plot_type='min'):
    """
    Plots a figure for one category across all countries focusing on minimum or maximum values.
    """
    for category in categories:
        fig, ax = plt.subplots(figsize=(5, 5))
        for country in countries:
            category_info = get_category_info(country=country, category=category, files=files)
            
            if plot_type == 'min':
                value = category_info.min_number
                label = f"{country} Min"
                filename = category_info.min_number_filename
            else:  # 'max'
                value = category_info.max_number
                label = f"{country} Max"
                filename = category_info.max_number_filename

           
            if filename is None:
                continue 
            traj = load_file(filename)
            radius_bins, pair_distribution = pp.compute_pair_distibution_function(traj_data=traj, radius_bin_size=0.1)
            ax.plot(radius_bins, pair_distribution, linewidth=0.9, label=country, color=colors[country])

            
        #ax.set_title(f"{category.capitalize()} Pair Distribution in {country}", fontsize=14)
        ax.set_xlabel(r"$r\; /\; m$", fontsize=14)
        ax.set_ylabel(r"$g(r)$", fontsize=14)
        ax.tick_params(axis='both', which='major', labelsize=12)
        #ax.grid(True, which='both', linestyle='--', alpha=0.3)
        ax.set_xlim([-0.1, max_radius])
        ax.set_ylim([-0.1, max_g])
        #ax.set_title(f"{category.capitalize()} ({'Minimum' if plot_type == 'min' else 'Maximum'})", fontsize=14)
        ax.legend(fontsize=14)
        fignames = [f"category/pdf/category_{category}_{plot_type}_across_countries.pdf",
                    f"category/png/category_{category}_{plot_type}_across_countries.png"]
        for figname in fignames: 
            plt.savefig(figname)    
        plt.close(fig)


#plot_category_info_min_max(countries, categories, files, plot_type='min')
plot_category_info_min_max(countries, categories, files, plot_type='max')
